# ASR v4 — Whisper Small LoRA r32 trên Colab GPU
Train, validation và test đều chạy trên Colab GPU. Train chỉ dùng train v4; checkpoint/early stopping chỉ dùng validation WER rồi CER; model được khóa trước khi test. V4 tái sử dụng đúng ba split v3 nên đây là so sánh cùng test set, không phải holdout mới. Trước khi chạy: **Runtime → Change runtime type → T4 GPU**.

In [3]:
from google.colab import drive
from pathlib import Path
import csv, json, os, shutil, subprocess, sys
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
else:
    print('Google Drive đã được mount')
REPO_URL = 'https://github.com/TiiAyyLuvBear/VoiceStudy-Assistant.git'
BRANCH = 'master'  # branch mặc định hiện tại của repository
RUN_NAME = 'whisper-small-lora-wide-v4-run1'  # đổi nếu chạy thí nghiệm mới
PROJECT_ROOT = Path('/content/VoiceStudy-Assistant')
RUN_ROOT = Path('/content/drive/MyDrive/VoiceStudy-Assistant-Colab') / RUN_NAME
MODEL_ROOT = RUN_ROOT / 'models/experimental/asr/v4'
REPORT_ROOT = RUN_ROOT / 'reports/asr/v4'
if (PROJECT_ROOT / '.git').is_dir():
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'], check=True)
elif PROJECT_ROOT.exists():
    raise RuntimeError(f'{PROJECT_ROOT} tồn tại nhưng không phải Git repo')
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(PROJECT_ROOT)], check=True)
os.chdir(PROJECT_ROOT)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print('Output sẽ lưu tại:', RUN_ROOT)

Google Drive đã được mount
Output sẽ lưu tại: /content/drive/MyDrive/VoiceStudy-Assistant-Colab/whisper-small-lora-wide-v4-run1


In [4]:
# Preflight GPU, dữ liệu v4 và kết quả tham chiếu original/v3.
import torch
from src.utils import canonical_csv_sha256
assert torch.cuda.is_available(), 'Hãy bật T4 GPU trong Runtime settings'
print('GPU:', torch.cuda.get_device_name(0))
manifest_path = Path('data/processed/v4/asr_finetune_manifest.json')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert manifest['dataset_version'] == 'v4' and manifest['freeze_status'] == 'DEVELOPMENT'
for split, expected in {'train': 1465, 'validation': 322, 'test': 249}.items():
    path = Path(f'data/processed/v4/metadata/asr_finetune_{split}.csv')
    with path.open(encoding='utf-8-sig', newline='') as stream:
        rows = list(csv.DictReader(stream))
    missing = [row['audio_path'] for row in rows if not Path(row['audio_path']).is_file()]
    assert len(rows) == expected and not missing, (split, len(rows), len(missing))
    assert canonical_csv_sha256(path) == manifest['datasets'][split]['canonical_csv_sha256']
    print(split, len(rows), 'rows — OK')
for path in [Path('reports/asr/v3/baseline_original/baseline_original_metrics.json'), Path('reports/asr/v3/final_test/test_metrics.json')]:
    assert path.is_file(), f'Thiếu file cần push lên Git: {path}'
from huggingface_hub import snapshot_download
BASE_MODEL = Path('/content/models/openai-whisper-small')
snapshot_download(repo_id='openai/whisper-small', local_dir=BASE_MODEL)
print('Base model:', BASE_MODEL)

GPU: Tesla T4
train 1465 rows — OK
validation 322 rows — OK
test 249 rows — OK


Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

flax_model.msgpack:   0%|          | 0.00/967M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

Base model: /content/models/openai-whisper-small


In [9]:
!pip uninstall -y torchao

In [10]:
!pip show torchao

In [11]:
diag_root = Path('/content/asr_v4_diagnostic')

cmd = [
      sys.executable, '-m', 'scripts.finetune_asr_v4',
      '--base-model', str(BASE_MODEL),
      '--output-root', str(diag_root),
      '--device', 'cuda',
      '--mixed-precision', 'fp16',
      '--batch-size', '1',
      '--gradient-accumulation', '1',
      '--validation-batch-size', '1',
      '--min-epochs', '1',
      '--max-epochs', '1',
      '--max-steps', '1',
      '--limit-train', '1',
      '--limit-validation', '1',
  ]

result = subprocess.run(
      cmd,
      text=True,
      stdout=subprocess.PIPE,
      stderr=subprocess.STDOUT,
  )

print(result.stdout)
print("RETURN CODE:", result.returncode)

/content/VoiceStudy-Assistant/scripts/finetune_asr_v4.py:278: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
epoch=1 step=1/1 loss=11.371257
validation_decode batch=1 samples=1/1
{"epoch": 1, "global_step": 1, "train_loss": 11.371256828308105, "validation_loss": 9.103666305541992, "validation": {"word_edits": 7, "reference_words": 6, "char_edits": 19, "reference_chars": 19, "wer": 1.1666666666666667, "cer": 1.0}, "improved": true, "non_improving_epochs": 0}
{
  "schema_version": 1,
  "status": "TRAINED_VALIDATION_SELECTED_NOT_TESTED",
  "dataset_version": "asr-v4",
  "method": "decoder-only wider LoRA on openai/whisper-small",
  "

In [12]:
# TRAIN + VALIDATION: r32; q/k/v/out + fc1/fc2; 4–6 epoch.
# Batch 4 + accumulation 4 cho effective batch 16 trên T4 16 GB.
training_summary = MODEL_ROOT / 'training_summary.json'
if training_summary.exists():
    print('Đã train xong, bỏ qua:', training_summary)
elif (MODEL_ROOT / 'best_adapter').exists():
    raise RuntimeError('Run bị gián đoạn; đổi RUN_NAME để chạy sạch, không ghi đè checkpoint')
else:
    subprocess.run([
        sys.executable, '-m', 'scripts.finetune_asr_v4',
        '--base-model', str(BASE_MODEL), '--output-root', str(MODEL_ROOT),
        '--device', 'cuda', '--mixed-precision', 'fp16',
        '--batch-size', '4', '--gradient-accumulation', '4',
        '--validation-batch-size', '8',
        '--min-epochs', '4', '--max-epochs', '6',
        '--early-stopping-patience', '1'
    ], check=True)

In [16]:

tokenizer_source = merged_dir / "tokenizer.json"
tokenizer_target = ct2_dir / "tokenizer.json"

print("Nguồn:", tokenizer_source, tokenizer_source.exists())
print("Đích:", tokenizer_target)

if not tokenizer_source.exists():
    raise FileNotFoundError(f"Không tìm thấy {tokenizer_source}")

shutil.copy2(tokenizer_source, tokenizer_target)

print("Đã chép:", tokenizer_target)
print("Kích thước:", tokenizer_target.stat().st_size)

Nguồn: /content/drive/MyDrive/VoiceStudy-Assistant-Colab/whisper-small-lora-wide-v4-run1/models/experimental/asr/v4/hf_merged/tokenizer.json True
Đích: /content/drive/MyDrive/VoiceStudy-Assistant-Colab/whisper-small-lora-wide-v4-run1/models/experimental/asr/v4/ctranslate2/tokenizer.json
Đã chép: /content/drive/MyDrive/VoiceStudy-Assistant-Colab/whisper-small-lora-wide-v4-run1/models/experimental/asr/v4/ctranslate2/tokenizer.json
Kích thước: 3930661


In [17]:
subprocess.run([
      sys.executable, "-m", "scripts.lock_asr_v4_model",
      "--model-dir", str(ct2_dir),
      "--training-summary", str(training_summary),
      "--export-summary", str(MODEL_ROOT / "export_summary.json"),
      "--output", str(lock_path),
      "--device", "cuda",
      "--compute-type", "float16",
  ], check=True)

print("Locked:", lock_path)

Locked: /content/drive/MyDrive/VoiceStudy-Assistant-Colab/whisper-small-lora-wide-v4-run1/models/experimental/asr/v4/locked_model.json


In [18]:
lock_cmd = [
      sys.executable, '-m', 'scripts.lock_asr_v4_model',
      '--model-dir', str(ct2_dir),
      '--training-summary', str(training_summary),
      '--export-summary', str(MODEL_ROOT / 'export_summary.json'),
      '--output', str(lock_path),
      '--device', 'cuda',
      '--compute-type', 'float16',
  ]

result = subprocess.run(
      lock_cmd,
      text=True,
      stdout=subprocess.PIPE,
      stderr=subprocess.STDOUT,
)

print(result.stdout)
print("RETURN CODE:", result.returncode)

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/VoiceStudy-Assistant/scripts/lock_asr_v4_model.py", line 129, in <module>
    raise SystemExit(main())
                     ^^^^^^
  File "/content/VoiceStudy-Assistant/scripts/lock_asr_v4_model.py", line 59, in main
    raise FileExistsError(f"Lock already exists: {args.output}")
FileExistsError: Lock already exists: /content/drive/MyDrive/VoiceStudy-Assistant-Colab/whisper-small-lora-wide-v4-run1/models/experimental/asr/v4/locked_model.json

RETURN CODE: 1


In [19]:
# Export checkpoint validation tốt nhất, convert CTranslate2, rồi KHÓA trước test.
merged_dir = MODEL_ROOT / 'hf_merged'
ct2_dir = MODEL_ROOT / 'ctranslate2'
lock_path = MODEL_ROOT / 'locked_model.json'
if not merged_dir.exists():
    subprocess.run([sys.executable, '-m', 'scripts.export_asr_v4',
        '--base-model', str(BASE_MODEL), '--adapter', str(MODEL_ROOT / 'best_adapter'),
        '--training-summary', str(training_summary), '--output-dir', str(merged_dir)], check=True)
if not ct2_dir.exists():
    subprocess.run(['ct2-transformers-converter', '--model', str(merged_dir),
        '--output_dir', str(ct2_dir), '--quantization', 'float16'], check=True)
if not lock_path.exists():
    subprocess.run([sys.executable, '-m', 'scripts.lock_asr_v4_model',
        '--model-dir', str(ct2_dir), '--training-summary', str(training_summary),
        '--export-summary', str(MODEL_ROOT / 'export_summary.json'), '--output', str(lock_path),
        '--device', 'cuda', '--compute-type', 'float16'], check=True)
print('Locked:', lock_path)

Locked: /content/drive/MyDrive/VoiceStudy-Assistant-Colab/whisper-small-lora-wide-v4-run1/models/experimental/asr/v4/locked_model.json


In [20]:
# TEST v4 đúng một lần sau khi lock.
test_dir = REPORT_ROOT / 'final_test'
test_metrics = test_dir / 'test_metrics.json'
if test_metrics.exists():
    print('Test v4 đã tồn tại; không chạy lại:', test_metrics)
else:
    subprocess.run([sys.executable, '-m', 'scripts.evaluate_faster_whisper_v4',
        '--model-dir', str(ct2_dir), '--lock', str(lock_path),
        '--output-dir', str(test_dir), '--confirm-test'], check=True)


In [21]:
# So sánh ba model, freeze manifest và in WER/CER.
comparison_path = REPORT_ROOT / 'comparison.json'
final_manifest = MODEL_ROOT / 'final_manifest.json'
if not comparison_path.exists():
    subprocess.run([sys.executable, '-m', 'scripts.finalize_asr_v4',
        '--v4', str(test_metrics), '--lock', str(lock_path),
        '--training-summary', str(training_summary),
        '--comparison', str(comparison_path), '--final-manifest', str(final_manifest)], check=True)
shutil.copy2('data/processed/v4/asr_finetune_manifest.json', RUN_ROOT / 'asr_finetune_manifest_frozen.json')
comparison = json.loads(comparison_path.read_text(encoding='utf-8'))
print('Kết quả cuối:', comparison_path)
for model in comparison['models']:
    print(model['label'], 'WER={:.2f}%'.format(model['wer'] * 100), 'CER={:.2f}%'.format(model['cer'] * 100))

Kết quả cuối: /content/drive/MyDrive/VoiceStudy-Assistant-Colab/whisper-small-lora-wide-v4-run1/reports/asr/v4/comparison.json
Whisper Small original (model used by ASR v2; no fine-tune) WER=23.21% CER=14.62%
Whisper Small LoRA v3 (r8, q_proj/v_proj) WER=20.00% CER=12.01%
Whisper Small wider LoRA v4 (r32, attention + fc1/fc2) WER=18.03% CER=11.76%
